# Gikomba Stall Analysis
**Analyst:** Daniel Kinara Osoro | **Cohort:** 1 Million Devs Cohort


Wanjiku runs a fresh-produce stall at Gikomba Market. This notebook uses only core Python to answer: How is the business doing? Is it growing? What should she restock, and when should she focus her effort?


In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Summary statistics
I use `sum()`, `max()` and `min()` for the totals. `list.index()` gives the position of the best and worst days. Positions 0–6 are week 1, so `position // 7 + 1` gives the week number.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest = max(daily_sales)
lowest = min(daily_sales)

best_pos = daily_sales.index(highest)
worst_pos = daily_sales.index(lowest)
best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales : KES {total_sales:,}")
print(f"Average     : KES {average_sales:,.2f}")
print(f"Highest     : KES {highest:,} ({best_day}, week {best_week})")
print(f"Lowest      : KES {lowest:,} ({worst_day}, week {worst_week})")

Total sales : KES 46,450
Average     : KES 3,317.86
Highest     : KES 5,600 (Sat, week 2)
Lowest      : KES 1,950 (Tue, week 1)


## Task 2: Week totals and growth
`week_totals()` slices the list into two weeks and returns both totals as a tuple. Growth is (week 2 − week 1) ÷ week 1 × 100.

In [3]:
def week_totals(sales):
    """Return (week 1 total, week 2 total) from a list of 14 daily sales."""
    week1 = sum(sales[:7])
    week2 = sum(sales[7:14])
    return (week1, week2)

week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total: KES {week1_total:,}")
print(f"Week 2 total: KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:.2f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: 10.66%


## Task 3: Classify each day
A day is "Strong" if it is more than 20% above the average, "Weak" if more than 20% below, otherwise "Normal". I apply the function to every day and count the results with a dictionary.

In [4]:
def classify_day(amount, average):
    """Return 'Strong' (>20% above average), 'Weak' (>20% below) or 'Normal'."""
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    else:
        return "Normal"

day_counts = {"Strong": 0, "Normal": 0, "Weak": 0}
for amount in daily_sales:
    label = classify_day(amount, average_sales)
    day_counts[label] = day_counts.get(label, 0) + 1

print(day_counts)

{'Strong': 4, 'Normal': 4, 'Weak': 6}


## Task 4: Weekdays vs weekends
I loop through each day name with its sales (`zip`), put Sat/Sun amounts in one list and the rest in another, then compare the averages.

In [5]:
weekday_sales = []
weekend_sales = []
for day, amount in zip(days, daily_sales):
    if day in ("Sat", "Sun"):
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_uplift_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average: KES {weekday_avg:,.2f}")
print(f"Weekend average: KES {weekend_avg:,.2f}")
print(f"Weekends are {weekend_uplift_pct:.2f}% higher than weekdays")

Weekday average: KES 2,685.00
Weekend average: KES 4,900.00
Weekends are 82.50% higher than weekdays


## Task 5: Stock reorder and gross margin
`products_to_reorder()` loops over `stock.items()` and collects products with quantity at or below the reorder level. Gross margin % = (price − cost) ÷ price × 100.

In [6]:
def products_to_reorder(stock_data):
    """Return a list of products whose qty is at or below their reorder level."""
    low_stock = []
    for product, info in stock_data.items():
        if info["qty"] <= info["reorder"]:
            low_stock.append(product)
    return low_stock

def gross_margin_pct(price, cost):
    """Return gross margin as a percentage: (price - cost) / price * 100."""
    return (price - cost) / price * 100

reorder_list = products_to_reorder(stock)
margins = {product: gross_margin_pct(info["price"], info["cost"])
           for product, info in stock.items()}

print("Reorder now:", reorder_list)
for product, margin in margins.items():
    print(f"{product:<10} {margin:.2f}%")

Reorder now: ['Tomatoes', 'Sukuma']
Tomatoes   33.33%
Onions     30.00%
Sukuma     40.00%
Potatoes   33.33%
Avocado    42.86%


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Task 6: Stall Performance Report
All results are printed using f-strings. Money uses the `,` format for thousands separators, and `<` / `>` for column alignment.

In [8]:
line = "=" * 46
print(line)
print("STALL PERFORMANCE REPORT".center(46))
print("Wanjiku's Fresh Produce, Gikomba Market".center(46))
print(line)

print("\nSALES SUMMARY (14 days)")
print(f"{'Total sales':<26}KES {total_sales:>12,}")
print(f"{'Average daily sales':<26}KES {average_sales:>12,.2f}")
print(f"{'Best day':<26}KES {highest:>12,}  ({best_day}, wk {best_week})")
print(f"{'Worst day':<26}KES {lowest:>12,}  ({worst_day}, wk {worst_week})")

print("\nGROWTH")
print(f"{'Week 1 total':<26}KES {week1_total:>12,}")
print(f"{'Week 2 total':<26}KES {week2_total:>12,}")
print(f"{'Week-on-week growth':<26}{growth_pct:>16.2f}%")

print("\nDAY PERFORMANCE")
for label, count in day_counts.items():
    print(f"{label:<26}{count:>16} days")

print("\nWEEKDAYS vs WEEKENDS")
print(f"{'Weekday average':<26}KES {weekday_avg:>12,.2f}")
print(f"{'Weekend average':<26}KES {weekend_avg:>12,.2f}")
print(f"{'Weekend uplift':<26}{weekend_uplift_pct:>16.2f}%")

print("\nSTOCK STATUS")
print(f"{'Product':<12}{'Qty':>5}{'Reorder':>9}{'Margin':>10}  Status")
for product, info in stock.items():
    status = "REORDER" if product in reorder_list else "OK"
    print(f"{product:<12}{info['qty']:>5}{info['reorder']:>9}"
          f"{margins[product]:>9.2f}%  {status}")
print(line)

           STALL PERFORMANCE REPORT           
   Wanjiku's Fresh Produce, Gikomba Market    

SALES SUMMARY (14 days)
Total sales               KES       46,450
Average daily sales       KES     3,317.86
Best day                  KES        5,600  (Sat, wk 2)
Worst day                 KES        1,950  (Tue, wk 1)

GROWTH
Week 1 total              KES       22,050
Week 2 total              KES       24,400
Week-on-week growth                  10.66%

DAY PERFORMANCE
Strong                                   4 days
Normal                                   4 days
Weak                                     6 days

WEEKDAYS vs WEEKENDS
Weekday average           KES     2,685.00
Weekend average           KES     4,900.00
Weekend uplift                       82.50%

STOCK STATUS
Product       Qty  Reorder    Margin  Status
Tomatoes       18       25    33.33%  REORDER
Onions         40       20    30.00%  OK
Sukuma          9       15    40.00%  REORDER
Potatoes       55       30    33.33%  OK

## Recommendations for Wanjiku

1. **Restock Tomatoes and Sukuma immediately.** Both are below their reorder levels (Tomatoes 18 kg vs 25 kg; Sukuma 9 kg vs 15 kg), and a stock-out before the weekend would cost sales.
2. **Stock up before Friday and focus on weekends.** Weekend sales average KES 4,900 per day versus KES 2,685 on weekdays, which is about 82.5% higher, and all 4 "Strong" days were Saturdays or Sundays.
3. **Push Avocado, your highest-margin product.** Its margin is 42.86% (KES 60 profit per kg), well above Onions at 30.00%, so shelf space and promotion should favour it.
4. **Run weekday promotions.** 6 of 14 days were "Weak" (more than 20% below the KES 3,317.86 average), and the weakest day was a Tuesday at KES 1,950, so a Tuesday offer could lift it.
5. **Keep up the momentum.** Sales grew 10.66% from KES 22,050 in week 1 to KES 24,400 in week 2, so the business is growing and extra stock for weekends is justified.